# 01 — Analyse Exploratoire des Données (EDA)

**Objectif** : Explorer et visualiser le jeu de données Oscar + IMDb afin de mieux comprendre sa structure, ses distributions, et de formuler des hypothèses pour la modélisation.

**Dataset principal** : `Data/Processed/oscar_imdb_merged.csv`  
**Période couverte** : Cérémonies 2000–2022  
**Auteurs** : Anna, Keira, Robin, Jonathan


## 0. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# Style global
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 5)
PALETTE_WIN = {True: '#F4A261', False: '#457B9D'}  # orange = gagnant, bleu = nommé

# Chemins
ROOT = Path('..').resolve()
DATA_MERGED = ROOT / 'Data' / 'Processed' / 'oscar_imdb_merged.csv'
DATA_RAW_OSCAR = ROOT / 'Data' / 'Raw' / 'Scraping' / 'all_data_oscars.csv'

print(f'Répertoire racine : {ROOT}')
print(f'Fichier merged    : {DATA_MERGED.exists()}')
print(f'Fichier oscar raw : {DATA_RAW_OSCAR.exists()}')

## 1. Chargement & Aperçu des Données

In [ ]:
df = pd.read_csv(DATA_MERGED)

# Nettoyage minimal
df['winner'] = df['winner'].astype(bool)

print(f'Shape : {df.shape[0]} lignes × {df.shape[1]} colonnes')
print(f'Période : {df["year"].min()} → {df["year"].max()}')
print(f'Cérémonies : {df["ceremony_number"].nunique()}')
print(f'Films uniques : {df["tconst"].nunique()}')
print(f'Catégories : {df["category"].nunique()}')
print(f'Gagnants : {df["winner"].sum()} ({df["winner"].mean()*100:.1f}%)')
df.head()

In [ ]:
# Résumé statistique des variables numériques
df[['imdb_rating', 'imdb_votes', 'runtime_minutes', 'n_genres', 'n_cast']].describe().round(2)

In [ ]:
# Valeurs manquantes
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
pct = (missing / len(df) * 100).round(1)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(missing.index, pct.values, color='#E63946')
ax.bar_label(bars, labels=[f'{v}%' for v in pct.values], padding=3)
ax.set_xlabel('% de valeurs manquantes')
ax.set_title('Valeurs manquantes par colonne')
ax.set_xlim(0, pct.max() * 1.2)
plt.tight_layout()
plt.show()

## 2. Distribution des Nominations dans le Temps

In [ ]:
# Nominations et gagnants par année
yearly = df.groupby('year').agg(
    total_nominations=('winner', 'count'),
    winners=('winner', 'sum')
).reset_index()

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()

ax1.bar(yearly['year'], yearly['total_nominations'], color='#457B9D', alpha=0.7, label='Total nominations')
ax2.plot(yearly['year'], yearly['winners'], color='#F4A261', lw=2.5, marker='o', ms=5, label='Gagnants')

ax1.set_xlabel('Année')
ax1.set_ylabel('Nombre de nominations', color='#457B9D')
ax2.set_ylabel('Nombre de gagnants', color='#F4A261')
ax1.set_title('Nominations et gagnants aux Oscars par année (2000–2022)')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout()
plt.show()

## 3. Analyse par Catégorie

In [ ]:
# Top catégories par nombre de nominations
cat_stats = df.groupby('category').agg(
    nominations=('winner', 'count'),
    winners=('winner', 'sum')
).reset_index()
cat_stats['win_rate'] = cat_stats['winners'] / cat_stats['nominations']
cat_stats = cat_stats.sort_values('nominations', ascending=False).head(18)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Nombre de nominations
sns.barplot(data=cat_stats.sort_values('nominations', ascending=True),
            x='nominations', y='category', color='#457B9D', ax=axes[0])
axes[0].set_title('Nombre de nominations (top 18 catégories)')
axes[0].set_xlabel('Nominations')
axes[0].set_ylabel('')

# Taux de victoire (= 1/nb_nommés en théorie)
sns.barplot(data=cat_stats.sort_values('win_rate', ascending=True),
            x='win_rate', y='category', color='#F4A261', ax=axes[1])
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
axes[1].set_title('Taux de victoire par catégorie')
axes[1].set_xlabel('Taux de victoire')
axes[1].set_ylabel('')

plt.suptitle('Analyse des catégories aux Oscars', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 4. Note IMDb : Nommés vs Gagnants

In [ ]:
# Distribution des notes IMDb selon le statut gagnant/nommé
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramme
for win, label in [(False, 'Nommé'), (True, 'Gagnant')]:
    subset = df[df['winner'] == win]['imdb_rating'].dropna()
    axes[0].hist(subset, bins=25, alpha=0.6,
                 color=PALETTE_WIN[win], label=f'{label} (n={len(subset)})', density=True)
axes[0].set_xlabel('Note IMDb')
axes[0].set_ylabel('Densité')
axes[0].set_title('Distribution des notes IMDb')
axes[0].legend()

# Boxplot
df_plot = df[['imdb_rating', 'winner']].dropna()
df_plot['Statut'] = df_plot['winner'].map({True: 'Gagnant', False: 'Nommé'})
sns.boxplot(data=df_plot, x='Statut', y='imdb_rating',
            palette={'Gagnant': '#F4A261', 'Nommé': '#457B9D'}, ax=axes[1])
axes[1].set_title('Note IMDb : Nommés vs Gagnants')
axes[1].set_ylabel('Note IMDb')
axes[1].set_xlabel('')

plt.suptitle('Relation entre note IMDb et victoire aux Oscars', fontsize=14)
plt.tight_layout()
plt.show()

# Moyennes
print('Note IMDb moyenne — Nommés  :', round(df[df['winner']==False]['imdb_rating'].mean(), 3))
print('Note IMDb moyenne — Gagnants:', round(df[df['winner']==True]['imdb_rating'].mean(), 3))

## 5. Popularité (Votes IMDb)

In [ ]:
# Distribution du nombre de votes (log-scale)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for win, label in [(False, 'Nommé'), (True, 'Gagnant')]:
    subset = df[df['winner'] == win]['imdb_votes'].dropna()
    axes[0].hist(np.log1p(subset), bins=30, alpha=0.6,
                 color=PALETTE_WIN[win], label=f'{label}', density=True)
axes[0].set_xlabel('log(1 + Votes IMDb)')
axes[0].set_ylabel('Densité')
axes[0].set_title('Distribution des votes IMDb (log)')
axes[0].legend()

df_plot2 = df[['imdb_votes', 'winner']].dropna()
df_plot2['Statut'] = df_plot2['winner'].map({True: 'Gagnant', False: 'Nommé'})
df_plot2['log_votes'] = np.log1p(df_plot2['imdb_votes'])
sns.boxplot(data=df_plot2, x='Statut', y='log_votes',
            palette={'Gagnant': '#F4A261', 'Nommé': '#457B9D'}, ax=axes[1])
axes[1].set_title('Votes IMDb : Nommés vs Gagnants (log)')
axes[1].set_ylabel('log(1 + Votes)')
axes[1].set_xlabel('')

plt.suptitle('Popularité (votes IMDb) et victoire aux Oscars', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Analyse des Genres

In [ ]:
# Exploser les genres (un film peut avoir plusieurs genres)
df_genres = df.dropna(subset=['genres']).copy()
df_genres['genre_list'] = df_genres['genres'].str.split(',')
df_exploded = df_genres.explode('genre_list')
df_exploded['genre_list'] = df_exploded['genre_list'].str.strip()

genre_stats = df_exploded.groupby('genre_list').agg(
    count=('winner', 'count'),
    win_rate=('winner', 'mean')
).reset_index()
genre_stats = genre_stats[genre_stats['count'] >= 10].sort_values('count', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Fréquence des genres parmi les nommés
top_genres = genre_stats.sort_values('count', ascending=True).tail(15)
sns.barplot(data=top_genres, x='count', y='genre_list', color='#457B9D', ax=axes[0])
axes[0].set_title('Genres les plus fréquents chez les nommés')
axes[0].set_xlabel('Apparitions')
axes[0].set_ylabel('')

# Taux de victoire par genre
top_winrate = genre_stats.sort_values('win_rate', ascending=True).tail(15)
sns.barplot(data=top_winrate, x='win_rate', y='genre_list', color='#F4A261', ax=axes[1])
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
axes[1].set_title('Taux de victoire par genre')
axes[1].set_xlabel('Taux de victoire')
axes[1].set_ylabel('')

plt.suptitle('Analyse des genres cinématographiques aux Oscars', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Durée des Films (Runtime)

In [ ]:
# Filtrer les durées aberrantes
df_rt = df[(df['runtime_minutes'] >= 60) & (df['runtime_minutes'] <= 240)].copy()
df_rt['Statut'] = df_rt['winner'].map({True: 'Gagnant', False: 'Nommé'})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramme
for win, label in [(False, 'Nommé'), (True, 'Gagnant')]:
    subset = df_rt[df_rt['winner'] == win]['runtime_minutes']
    axes[0].hist(subset, bins=30, alpha=0.6, color=PALETTE_WIN[win], label=label, density=True)
axes[0].axvline(df_rt[df_rt['winner']==False]['runtime_minutes'].median(),
                color='#457B9D', ls='--', lw=1.5, label='Médiane nommés')
axes[0].axvline(df_rt[df_rt['winner']==True]['runtime_minutes'].median(),
                color='#F4A261', ls='--', lw=1.5, label='Médiane gagnants')
axes[0].set_xlabel('Durée (minutes)')
axes[0].set_ylabel('Densité')
axes[0].set_title('Distribution de la durée des films')
axes[0].legend(fontsize=8)

# Violin plot
sns.violinplot(data=df_rt, x='Statut', y='runtime_minutes',
               palette={'Gagnant': '#F4A261', 'Nommé': '#457B9D'}, ax=axes[1], inner='quartile')
axes[1].set_title('Durée des films : Nommés vs Gagnants')
axes[1].set_ylabel('Durée (minutes)')
axes[1].set_xlabel('')

plt.suptitle('Durée des films et victoire aux Oscars', fontsize=14)
plt.tight_layout()
plt.show()

print('Durée médiane — Nommés  :', df_rt[df_rt['winner']==False]['runtime_minutes'].median(), 'min')
print('Durée médiane — Gagnants:', df_rt[df_rt['winner']==True]['runtime_minutes'].median(), 'min')

## 8. Matrice de Corrélation des Variables Numériques

In [ ]:
# Sélection des features numériques
num_cols = ['winner', 'imdb_rating', 'imdb_votes', 'runtime_minutes', 'n_genres', 'n_cast']
df_corr = df[num_cols].copy()
df_corr['winner'] = df_corr['winner'].astype(int)
df_corr['log_votes'] = np.log1p(df_corr['imdb_votes'])
df_corr = df_corr.drop(columns='imdb_votes')  # remplacé par log_votes

corr_matrix = df_corr.corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, mask=mask,
            square=True, linewidths=.5, ax=ax)
ax.set_title('Matrice de corrélation des variables numériques', pad=15)
plt.tight_layout()
plt.show()

## 9. Évolution de la Note IMDb dans le Temps

In [ ]:
# Note IMDb moyenne des nommés et gagnants par année
rating_yearly = df.groupby(['year', 'winner'])['imdb_rating'].mean().reset_index()
rating_yearly['Statut'] = rating_yearly['winner'].map({True: 'Gagnant', False: 'Nommé'})

fig, ax = plt.subplots(figsize=(14, 5))
for win, label, color in [(False, 'Nommés', '#457B9D'), (True, 'Gagnants', '#F4A261')]:
    subset = rating_yearly[rating_yearly['winner'] == win]
    ax.plot(subset['year'], subset['imdb_rating'], marker='o', ms=5,
            color=color, lw=2, label=label)
    # Tendance
    z = np.polyfit(subset['year'], subset['imdb_rating'], 1)
    p = np.poly1d(z)
    ax.plot(subset['year'], p(subset['year']), ls='--', color=color, alpha=0.5, lw=1.5)

ax.set_xlabel('Année')
ax.set_ylabel('Note IMDb moyenne')
ax.set_title('Évolution de la note IMDb des nommés et gagnants (avec tendance)')
ax.legend()
plt.tight_layout()
plt.show()

## 10. Focus : Best Picture

In [ ]:
# Zoom sur la catégorie phare
bp = df[df['category'] == 'Best Picture'].copy()
bp['Statut'] = bp['winner'].map({True: 'Gagnant ✓', False: 'Nommé'})

print(f'Best Picture — {len(bp)} entrées, {bp["winner"].sum()} gagnants')
print(f'Note IMDb moyenne des gagnants Best Picture : {bp[bp["winner"]=="True"]["imdb_rating"].mean():.2f}')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Note IMDb
sns.boxplot(data=bp, x='Statut', y='imdb_rating',
            palette={'Gagnant ✓': '#F4A261', 'Nommé': '#457B9D'}, ax=axes[0])
axes[0].set_title('Note IMDb')
axes[0].set_xlabel('')

# Votes
bp['log_votes'] = np.log1p(bp['imdb_votes'])
sns.boxplot(data=bp, x='Statut', y='log_votes',
            palette={'Gagnant ✓': '#F4A261', 'Nommé': '#457B9D'}, ax=axes[1])
axes[1].set_title('Votes IMDb (log)')
axes[1].set_xlabel('')

# Durée
bp_rt = bp[(bp['runtime_minutes'] >= 60) & (bp['runtime_minutes'] <= 240)]
sns.boxplot(data=bp_rt, x='Statut', y='runtime_minutes',
            palette={'Gagnant ✓': '#F4A261', 'Nommé': '#457B9D'}, ax=axes[2])
axes[2].set_title('Durée (minutes)')
axes[2].set_xlabel('')

plt.suptitle('Best Picture — Comparaison Nommés vs Gagnants', fontsize=14)
plt.tight_layout()
plt.show()

## 11. Synthèse & Hypothèses pour la Modélisation

### Observations clés

| Variable | Observation | Signal prédictif |
|----------|-------------|------------------|
| `imdb_rating` | Les gagnants ont une note légèrement supérieure (~+0.1–0.2 pts) | Faible mais présent |
| `imdb_votes` | Les gagnants sont légèrement plus populaires (log-votes) | Faible |
| `runtime_minutes` | Les gagnants ont tendance à être plus longs | Modéré |
| `genre` | Drama, Biography, History dominent chez les gagnants | Fort |
| `category` | Le taux de victoire théorique est ~20% (5 nommés/catégorie) | Dépend du contexte |

### Prochaines étapes suggérées

1. **Feature engineering** : créer `is_drama`, `is_biography`, encoder les genres en one-hot
2. **Analyse par catégorie** : construire des modèles séparés par catégorie (Best Picture vs autres)
3. **Variables manquantes** : imputer ou exclure `n_cast`, `writers`
4. **Données externes** : enrichir avec box-office, scores critiques (Rotten Tomatoes, Metacritic)
5. **Baseline model** : régression logistique par catégorie avec les features actuelles
